Transformação de dados

In [1]:
import pandas as pd
from camara_deputados.ingestion.data_loader import DataLoader
from camara_deputados.extraction.write import DataWrite
from camara_deputados.transformer.transformer import DataTransformer
from camara_deputados.transformer.uritransformer import UriTransformer


In [2]:
# instâncias

bronze = DataLoader('bronze')
salva= DataWrite()
transforma = DataTransformer()



In [3]:
dfs_bronze = bronze.carregar_parquets()


📂 Lendo dados da camada bronze: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/notebooks/../data/bronze

✅ deputados_detalhamento: 513 linhas, 34 colunas
✅ deputados_frentes: 131244 linhas, 6 colunas
✅ deputados_lista: 513 linhas, 10 colunas
✅ frentes: 100 linhas, 5 colunas
✅ frentes_detalhamento: 100 linhas, 22 colunas
✅ partidos: 15 linhas, 5 colunas
✅ partidos_detalhamento: 15 linhas, 23 colunas
✅ proposicoes: 60 linhas, 10 colunas
✅ proposicoes_autores: 57 linhas, 8 colunas
✅ proposicoes_detalhamento: 60 linhas, 36 colunas
✅ proposicoes_temas: 62 linhas, 5 colunas
✅ proposicoes_votacoes: 271 linhas, 13 colunas
✅ tipos_proposicao: 544 linhas, 5 colunas
✅ votacoes_detalhamento: 271 linhas, 21 colunas
✅ votacoes_orientacao: 399 linhas, 7 colunas

🎯 Carregamento finalizado.


# Cria camada silver

## Deputado

In [4]:
# criando a silver Deputado

df_deputadoDetalhamento = dfs_bronze['deputados_detalhamento']

colunas_origem = list(df_deputadoDetalhamento)

In [5]:
# as colunas sinalizadas com # não subirão para a silver 

colunas_extraidas = ['id'
    #, 'uri'
    , 'nomeCivil'
    #, 'cpf'
    , 'sexo'
    #, 'urlWebsite'
    #, 'redeSocial'  
    , 'dataNascimento'
    , 'dataFalecimento'
    , 'ufNascimento'
    , 'municipioNascimento'
    , 'escolaridade'
    #, 'ultimoStatus.id'
    #, 'ultimoStatus.uri'
    #, 'ultimoStatus.nome'
    , 'ultimoStatus.siglaPartido'
    #, 'ultimoStatus.uriPartido'
    , 'ultimoStatus.siglaUf'
    , 'ultimoStatus.idLegislatura'
    #, 'ultimoStatus.urlFoto'
    , 'ultimoStatus.email'
    #, 'ultimoStatus.data'
    , 'ultimoStatus.nomeEleitoral'
    #, 'ultimoStatus.gabinete.nome'
    #, 'ultimoStatus.gabinete.sala'
    #, 'ultimoStatus.gabinete.andar'
    #, 'ultimoStatus.gabinete.telefone'
    #, 'ultimoStatus.gabinete.predio'
    #, 'ultimoStatus.gabinete.email'
    , 'ultimoStatus.situacao'
    , 'ultimoStatus.condicaoEleitoral'
    #, 'ultimoStatus.descricaoStatus'
    #, 'source_id'
    #, 'data_extracao'
    ]
df_s_deputado = df_deputadoDetalhamento[colunas_extraidas]

In [6]:
#renomeado as colunas 

mapeamento_deputados = {
    'id':('id_deputado','int')
    ,'nomeCivil':('nom_NomeCivil','str')
    ,'sexo':('nom_Sexo','str')
    ,'dataNascimento':('dat_DataNasc','date')
    ,'dataFalecimento':('dat_DataFalecimento','date')
    ,'ufNascimento':('nom_UFNasc','str')
    ,'municipioNascimento':('nom_MunicipioNasci','str')
    ,'escolaridade':('nom_Escolaridade','str')
    ,'ultimoStatus.siglaPartido':( 'nom_SiglaPartido','str')
    ,'ultimoStatus.siglaUf':('nom_UFRepresenta', 'str')
    ,'ultimoStatus.idLegislatura':('id_Legislatura','int')
    ,'ultimoStatus.email':('nom_Email','str')
    ,'ultimoStatus.nomeEleitoral':('nom_NomeEleitoral','str')
    ,'ultimoStatus.situacao':('nom_Situacao','str')
    ,'ultimoStatus.condicaoEleitoral':('nom_CondEleitoral','str')

}

df_s_deputado = transforma.rename_and_cast(df_s_deputado,mapeamento_deputados)

In [49]:
salva.save_parquet(df_s_deputado, 'silver_deputado', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_deputado/silver_deputado.parquet


## Proposicões Detalhada

In [4]:
#criando a tabela silver das proposições

df_proposicaoDetalhamento = dfs_bronze['proposicoes_detalhamento']

colunas_origem = list(df_proposicaoDetalhamento.columns)
print(colunas_origem)

['id', 'uri', 'siglaTipo', 'codTipo', 'numero', 'ano', 'ementa', 'dataApresentacao', 'uriOrgaoNumerador', 'uriAutores', 'descricaoTipo', 'ementaDetalhada', 'keywords', 'uriPropPrincipal', 'uriPropAnterior', 'uriPropPosterior', 'urlInteiroTeor', 'urnFinal', 'texto', 'justificativa', 'statusProposicao.dataHora', 'statusProposicao.sequencia', 'statusProposicao.siglaOrgao', 'statusProposicao.uriOrgao', 'statusProposicao.uriUltimoRelator', 'statusProposicao.regime', 'statusProposicao.descricaoTramitacao', 'statusProposicao.codTipoTramitacao', 'statusProposicao.descricaoSituacao', 'statusProposicao.codSituacao', 'statusProposicao.despacho', 'statusProposicao.url', 'statusProposicao.ambito', 'statusProposicao.apreciacao', 'source_id', 'data_extracao']


In [5]:
# as colunas sinalizadas com # não subirão para a silver

mapeamento_proposicao= {
    'id':('id_proposicao','int')
    #,'uri'
    ,'siglaTipo':('nom_TipoProposicao','str')
    ,'codTipo':('cod_Tipo','int')
    ,'numero':('num_NumeroProp','int')
    ,'ano':('num_ano', 'int')
    ,'ementa':('nom_Ementa','str')
    ,'dataApresentacao':('dat_Apresentacao','date')
    #,'uriOrgaoNumerador'
    ,'uriAutores':('uri_Autor','str')
    #,'descricaoTipo'
    #,'ementaDetalhada'
    ,'keywords':('nom_Keywords','str')
    #,'uriPropPrincipal'
    #,'uriPropAnterior'
    #,'uriPropPosterior'
    #,'urlInteiroTeor'
    #,'urnFinal'
    #,'texto'
    #,'justificativa'
    #,'statusProposicao.dataHora'
    #,'statusProposicao.sequencia'
    ,'statusProposicao.siglaOrgao':('nom_SiglaOrgao', 'str')
    #,'statusProposicao.uriOrgao'
    ,'statusProposicao.uriUltimoRelator':('uri_Relator', 'str')
    ,'statusProposicao.regime':('nom_regime','str')
    #,'statusProposicao.descricaoTramitacao'
    ,'statusProposicao.codTipoTramitacao':('cod_TipoTramitacao','int')
    #,'statusProposicao.descricaoSituacao'
    ,'statusProposicao.codSituacao':('cod_Situacao','int')
    #,'statusProposicao.despacho'
    ,'statusProposicao.url':('nom_LinkProposicao','str')
    #,'statusProposicao.ambito'
    #,'statusProposicao.apreciacao'
    #,'source_id'
    #,'data_extracao'
}

df_s_proposicao = transforma.rename_and_cast(df_proposicaoDetalhamento, mapeamento_proposicao)

In [6]:
# Pega o id da uri

colunas_uri = {'uri_Autor':('nom_TipoAutor','id_Autor'), 'uri_Relator':('nom_TipoRelator','id_Relator')}

df_s_proposicao = (UriTransformer(df_s_proposicao).extrair_tipos_e_ids(colunas_uri) .get_df())

df_s_proposicao = df_s_proposicao.drop(columns=colunas_uri)

In [7]:
# salva as proposições na silver

salva.save_parquet(df_s_proposicao, 'silver_proposica','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_proposica/silver_proposica.parquet


In [8]:
df_s_proposicao.columns

Index(['id_proposicao', 'uri', 'nom_TipoProposicao', 'cod_Tipo',
       'num_NumeroProp', 'num_ano', 'nom_Ementa', 'dat_Apresentacao',
       'uriOrgaoNumerador', 'descricaoTipo', 'ementaDetalhada', 'nom_Keywords',
       'uriPropPrincipal', 'uriPropAnterior', 'uriPropPosterior',
       'urlInteiroTeor', 'urnFinal', 'texto', 'justificativa',
       'statusProposicao.dataHora', 'statusProposicao.sequencia',
       'nom_SiglaOrgao', 'statusProposicao.uriOrgao', 'nom_regime',
       'statusProposicao.descricaoTramitacao', 'cod_TipoTramitacao',
       'statusProposicao.descricaoSituacao', 'cod_Situacao',
       'statusProposicao.despacho', 'nom_LinkProposicao',
       'statusProposicao.ambito', 'statusProposicao.apreciacao', 'source_id',
       'data_extracao', 'nom_TipoAutor', 'id_Autor', 'nom_TipoRelator',
       'id_Relator'],
      dtype='str')

## Autores Proposicões

In [16]:
df_proposicaoAutores = dfs_bronze['proposicoes_autores']

mapeamento_autores = list(df_proposicaoAutores.columns)

In [17]:
mapeamento_autores = {
    'uri': ('uri_Autor', 'str'),
    'nome': ('nom_Autor', 'str'),
    'codTipo': ('cod_TipoAutor', 'int'),
    'tipo': ('nom_TipoAutor', 'str'),
    'ordemAssinatura': ('num_OrdemAssinatura', 'int'),
    'proponente': ('ind_Proponente', 'str'),  
    'source_id': ('id_proposicao', 'int'),
    'data_extracao': ('dat_Extracao', 'date')
}

df_s_autores  = transforma.rename_and_cast(df_proposicaoAutores, mapeamento_autores)

## Resgata ID da uri

df_s_autores = (UriTransformer(df_s_autores).extrair_tipo_e_id('uri_Autor','nom_TipoAutor','id_Autor')).get_df()

In [18]:
salva.save_parquet(df_s_autores, 'silver_autores','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_autores/silver_autores.parquet


## Silver temas das proposicoes

In [26]:
df_proposicoesTemas = dfs_bronze['proposicoes_temas']

list_temas = list(df_proposicoesTemas.columns)

mapeamento_temas = {
    'codTema': ('cod_Tema', 'int'),
    'tema': ('nom_Tema', 'str'),
    'relevancia': ('num_Relevancia', 'int'),
    'source_id': ('id_proposicao', 'int'),
    'data_extracao': ('dat_Extracao', 'date')
}

df_s_temasProposicoes = transforma.rename_and_cast(df_proposicoesTemas,mapeamento_temas)

In [27]:
salva.save_parquet(df_s_temasProposicoes, 'silver_temasProposicao', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_temasProposicao/silver_temasProposicao.parquet


Criando as dimensões

In [55]:
# criando a dim_deputados
df_deputadoDetalhamento = bronze.carregar_tabela('deputados_detalhamento')

# Mapeia as colunas que irão para camada silver e nomeia
mapeamento = {'id':'id_deputado',
           'nomeCivil':'nom_Nome',
           'sexo':'nom_Sexo',
           'dataNascimento':'dat_DataNascimento',
           'ufNascimento':'nom_UF',
           'municipioNascimento':'nom_MunicipioNatal',
           'escolaridade':'nom_Escolaridade'
}

#Resgata as colunas originais
colunas_origem = list(mapeamento.keys())


#cria a  dim_deputado com as colunas renomeadas
dim_deputado = df_deputadoDetalhamento[colunas_origem].rename(columns=mapeamento)



✅ deputados_detalhamento: 513 linhas, 34 colunas


In [56]:
dim_deputado.sample(5)

,id_deputado,nom_Nome,nom_Sexo,dat_DataNascimento,nom_UF,nom_MunicipioNatal,nom_Escolaridade
466,204557,SIDNEY RICARDO DE OLIVEIRA LEITE,M,1967-04-08,AM,Manaus,Superior
423,73801,RENILDO VASCONCELOS CALHEIROS,M,1959-04-20,AL,Murici,Superior
376,220706,NELSON FERNANDO PADOVANI,M,1977-10-29,PR,Cascavel,Superior
259,214694,JORGE GOETTEN DE LIMA,M,1962-04-10,SC,Mirim Doce,Superior
141,160599,DIMAS FABIANO TOLEDO JÚNIOR,M,1973-05-23,RJ,Macaé,Superior


In [57]:
dim_deputado['dat_DataNascimento']=pd.to_datetime(dim_deputado['dat_DataNascimento'],errors='coerce')

dim_deputado = dim_deputado.sort_values("id_deputado").reset_index(drop=True)



In [58]:
# salvando na silver

salva.save_parquet(dim_deputado, 'dim_s_deputados', layer='gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_deputados/dim_s_deputados.parquet


In [59]:
## Dim_frentes


df_frentes = bronze.carregar_tabela('frentes')

mapeamento_frentes = {'id':'id_frentes',
                      'titulo':'nom_TituloFrente',
                      'idLegislatura':'num_Legislatura'

}

colunas_origem = list(mapeamento_frentes.keys())

dim_frente = df_frentes[colunas_origem].rename(columns=mapeamento_frentes)

✅ frentes: 100 linhas, 5 colunas


In [60]:
dim_frente.sample(5)

,id_frentes,nom_TituloFrente,num_Legislatura
91,55570,Frente Parlamentar do Desporto Escolar,57
88,54563,"Frente Parlamentar em Apoio ao Petróleo, Gás e...",57
85,55568,Frente Parlamentar Mista da Medicina,57
4,55710,Frente Parlamentar Mista Brasil - Espanha,57
78,54560,"Frente Parlamentar Mista em Defesa da Criança,...",57


In [61]:
# salva na camada gold_temp 

salva.save_parquet(dim_frente, 'dim_s_frente', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_frente/dim_s_frente.parquet


In [62]:
# Cria dim_partido na gold_temp

df_partidos = bronze.carregar_tabela('partidos')

mapeamento_partidos = {'id':'id_partido',
                       'sigla':'nom_Sigla',
                       'nome':'nom_NomePartido'}

colunas_origem = list(mapeamento_partidos.keys())

dim_partido = df_partidos[colunas_origem].rename(columns=mapeamento_partidos)

✅ partidos: 15 linhas, 5 colunas


In [63]:
dim_partido.sample(5)

,id_partido,nom_Sigla,nom_NomePartido
14,36839,PSOL,Partido Socialismo e Liberdade
9,37903,PP,Progressistas
2,36899,MDB,Movimento Democrático Brasileiro
4,37901,NOVO,Partido Novo
6,36786,PDT,Partido Democrático Trabalhista


In [64]:
salva.save_parquet(dim_partido,'dim_s_partido','gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_partido/dim_s_partido.parquet


In [65]:
# Criando a dim_proposicao

df_proposicao = bronze.carregar_tabela('proposicoes_detalhamento')

mapeamento_proposicao = {
    'id':'id_proposicao',
    'codTipo':'cod_Tipo',
    'numero':'num_NumeroProposicao',
    'ano':'num_Ano',
    'ementa':'nom_Ementa',
    'keywords':'nom_Keywords'
}

colunas_origem = list(mapeamento_proposicao.keys())

dim_proposicao = df_proposicao[colunas_origem].rename(columns = mapeamento_proposicao)

✅ proposicoes_detalhamento: 60 linhas, 36 colunas


In [67]:
salva.save_parquet(dim_proposicao,'dim_s_proposicao', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_proposicao/dim_s_proposicao.parquet


Criando a dim_tema a partir da tabela proposicoes Temas

In [ ]:
#Carregando a proposicoe_temas

df_tema = bronze.carregar_tabela('proposicoes_temas')

✅ proposicoes_temas: 62 linhas, 5 colunas


In [68]:
dim_tema = (
    df_tema[['codTema', 'tema']]
    .drop_duplicates()
    .dropna(subset=['codTema'])
    .sort_values('codTema')
    .reset_index(drop=True)
)

In [69]:
display(dim_tema)

,codTema,tema
0,34,Administração Pública
1,37,Comunicações
2,40,Economia
3,41,Cidades e Desenvolvimento Urbano
4,42,Direito Civil e Processual Civil
5,43,Direito Penal e Processual Penal
6,44,Direitos Humanos e Minorias
7,46,Educação
8,48,Meio Ambiente e Desenvolvimento Sustentável
9,52,Previdência e Assistência Social


In [70]:
# renomeando e salvando na silver 
 
mapeamento_temas = {
    'codTema':'cod_Tema',
    'tema':'nom_Tema'
}

colunas_origem = list(mapeamento_temas.keys())

dim_tema = dim_tema[colunas_origem].rename(columns=mapeamento_temas)

salva.save_parquet(dim_tema, 'dim_s_tema', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_tema/dim_s_tema.parquet


In [71]:
def analisar_dfs(dfs):
    resultado = []

    for nome_tabela, df in dfs.items():
        for coluna in df.columns:
            
            total = len(df)
            nulos = df[coluna].isnull().sum()
            
            resultado.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "tipo": df[coluna].dtype,
                "nulos": nulos,
                "%_nulos": nulos / total,
                "unicos": df[coluna].astype(str).nunique(),
                "total_linhas": total
            })

    return pd.DataFrame(resultado)

In [72]:
dfs_dim = {
    'dim_deputado':dim_deputado,
    'dim_frente':dim_frente,
    'dim_partido':dim_partido,
    'dim_proposicacao':dim_proposicao,
    'dim_tema':dim_tema

}

In [73]:
df_analiseDim = analisar_dfs(dfs_dim)

In [74]:
display(df_analiseDim)

,tabela,coluna,tipo,nulos,%_nulos,unicos,total_linhas
0,dim_deputado,id_deputado,int64,0,0.000000,513,513
1,dim_deputado,nom_Nome,str,0,0.000000,513,513
2,dim_deputado,nom_Sexo,str,0,0.000000,2,513
3,dim_deputado,dat_DataNascimento,datetime64[us],0,0.000000,502,513
4,dim_deputado,nom_UF,str,1,0.001949,27,513
5,dim_deputado,nom_MunicipioNatal,str,2,0.003899,267,513
6,dim_deputado,nom_Escolaridade,str,12,0.023392,11,513
7,dim_deputado,data_extracao,datetime64[us],0,0.000000,1,513
8,dim_frente,id_frentes,int64,0,0.000000,100,100
9,dim_frente,nom_TituloFrente,str,0,0.000000,100,100
